In [2]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
import os

# --- CONFIGURATION ---
INPUT_FILE = "test_track_a.jsonl"
OUTPUT_FILE = "track_a.jsonl"
MODEL_NAME = 'all-MiniLM-L6-v2'  # A fast and powerful model for similarity

# --- STEP 1: SETUP ---
print("Installing sentence-transformers (this takes a moment)...")
!pip install -q sentence-transformers

print(f"Loading model: {MODEL_NAME}...")
# Load the model onto the GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Model loaded on {device.upper()}!")

# --- STEP 2: LOAD DATA ---
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Please upload {INPUT_FILE} to the Colab files sidebar!")

print(f"Reading {INPUT_FILE}...")
df = pd.read_json(INPUT_FILE, lines=True)

# --- STEP 3: COMPUTE EMBEDDINGS ---
# We encode all stories into vectors. This captures their *meaning*.
print(f"Encoding {len(df)} stories (Anchor, A, and B)...")

# Encode all lists at once for speed
anchors = model.encode(df['anchor_text'].tolist(), convert_to_tensor=True, show_progress_bar=True)
stories_a = model.encode(df['text_a'].tolist(), convert_to_tensor=True, show_progress_bar=True)
stories_b = model.encode(df['text_b'].tolist(), convert_to_tensor=True, show_progress_bar=True)

# --- STEP 4: PREDICT ---
print("Calculating semantic similarities...")
results = []

# Compute Cosine Similarity for the entire batch
# This returns a list of scores for (Anchor vs A) and (Anchor vs B)
scores_a = util.cos_sim(anchors, stories_a).diagonal()
scores_b = util.cos_sim(anchors, stories_b).diagonal()

for i in range(len(df)):
    # The Tensor contains the score, so we extract the float value
    score_a_val = scores_a[i].item()
    score_b_val = scores_b[i].item()

    # Prediction logic: True if A is closer, else False
    prediction = True if score_a_val > score_b_val else False

    results.append({
        "id": i, # Changed from df.iloc[i]['id'] to i
        "text_a_is_closer": prediction
    })

# --- STEP 5: SAVE SUBMISSION ---
print("Saving results...")
output_df = pd.DataFrame(results)
output_df.to_json(OUTPUT_FILE, orient='records', lines=True)

# Zip it for CodaBench
os.system(f"zip -j submission_neural.zip {OUTPUT_FILE}")

print(f"🎉 Success! Download 'submission_neural.zip' and upload it to CodaBench.")

Installing sentence-transformers (this takes a moment)...
Loading model: all-MiniLM-L6-v2...
Model loaded on CPU!
Reading test_track_a.jsonl...
Encoding 400 stories (Anchor, A, and B)...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating semantic similarities...
Saving results...
🎉 Success! Download 'submission_neural.zip' and upload it to CodaBench.
